In [8]:
# Install required packages
!pip install lightgbm catboost optuna torch geohash2 tqdm

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 30.0 MB/s eta 0:00:00
  Created wheel for geohash2: filename=geohash2-1.1-py3-none-any.whl size=15543 sha256=7a6315fd99aa0f83d8e894f4094814a399c65cef211a60a43036cd88918381f5
  Stored in directory: /root/.cache/pip/wheels/00/d5/b6/3fbe4088f7912982f596eaddfd593d16096468a2f13e470ae7
Successfully built geohash2


In [9]:
# ==============================================================================
# COMPLETE GRIDLOCK HACKATHON SOLUTION - ADVANCED ENSEMBLE (FIXED)
# ==============================================================================

import pandas as pd
import numpy as np
import warnings
import gc
from datetime import datetime

warnings.filterwarnings('ignore')

# Set seeds
SEED = 42
np.random.seed(SEED)
import random
random.seed(SEED)

# Machine Learning
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor,
                              AdaBoostRegressor, ExtraTreesRegressor)
from sklearn.linear_model import Ridge

# LightGBM
import lightgbm as lgb

# CatBoost
from catboost import CatBoostRegressor

# XGBoost
try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False
    print("XGBoost not installed. Run: pip install xgboost")

# Geohash
import geohash2

print("="*80)
print("GRIDLOCK HACKATHON - ADVANCED ENSEMBLE SOLUTION (FIXED)")
print("="*80)

GRIDLOCK HACKATHON - ADVANCED ENSEMBLE SOLUTION (FIXED)


In [10]:
# ==============================================================================
# DATA PROCESSING CLASS
# ==============================================================================

class DataProcessor:
    """Load and prepare data"""

    @staticmethod
    def load_data(train_path, test_path):
        df_train = pd.read_csv(train_path)
        df_test = pd.read_csv(test_path)
        return df_train, df_test

    @staticmethod
    def fix_data_types(df):
        df = df.copy()

        numeric_cols = ['NumberofLanes', 'Temperature', 'day']
        for col in numeric_cols:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')

        if 'LargeVehicles' in df.columns:
            lv_map = {'true': 1, 'yes': 1, '1': 1, 'True': 1, 'TRUE': 1,
                     'false': 0, 'no': 0, '0': 0, 'False': 0, 'FALSE': 0}
            df['LargeVehicles'] = df['LargeVehicles'].astype(str).str.lower().map(lv_map).fillna(0).astype(int)

        if 'Landmarks' in df.columns:
            lm_map = {'true': 1, 'yes': 1, '1': 1, 'True': 1, 'TRUE': 1,
                     'false': 0, 'no': 0, '0': 0, 'False': 0, 'FALSE': 0}
            df['Landmarks'] = df['Landmarks'].astype(str).str.lower().map(lm_map).fillna(0).astype(int)

        return df

    @staticmethod
    def parse_timestamps(df, base_year=2024):
        df = df.copy()

        def fix_time_format(t):
            if pd.isna(t) or str(t).strip() == '' or str(t).lower() == 'nan':
                return np.nan
            parts = str(t).split(':')
            if len(parts) == 2:
                return f"{int(parts[0]):02d}:{int(parts[1]):02d}"
            return np.nan

        df['timestamp'] = df['timestamp'].astype(str).str.strip().apply(fix_time_format)

        base_date = pd.Timestamp(f"{base_year}-01-01")
        df['date_part'] = base_date + pd.to_timedelta(df['day'] - 1, unit='D')
        df['datetime_str'] = df['date_part'].dt.strftime('%Y-%m-%d') + ' ' + df['timestamp']
        df['timestamp'] = pd.to_datetime(df['datetime_str'], format='%Y-%m-%d %H:%M', errors='coerce')

        df = df.dropna(subset=['timestamp']).reset_index(drop=True)
        df = df.drop(columns=['datetime_str', 'date_part'])

        return df

    @staticmethod
    def handle_missing_values(df, train_df=None):
        df = df.copy()

        if 'Temperature' in df.columns:
            df['Temperature'] = pd.to_numeric(df['Temperature'], errors='coerce')

        cat_cols = ['RoadType', 'Weather', 'Landmarks', 'LargeVehicles']
        for col in cat_cols:
            if col in df.columns:
                df[col] = df[col].fillna('Missing').astype(str)

        if 'Temperature' in df.columns:
            if train_df is not None and 'Temperature' in train_df.columns:
                train_temp = pd.to_numeric(train_df['Temperature'], errors='coerce')
                fill_value = train_temp.median()
            else:
                fill_value = df['Temperature'].median()

            df['Temperature'] = df['Temperature'].fillna(fill_value)
            df['Temperature'] = pd.to_numeric(df['Temperature'], errors='coerce').fillna(fill_value)

        if 'NumberofLanes' in df.columns:
            df['NumberofLanes'] = pd.to_numeric(df['NumberofLanes'], errors='coerce')
            df['NumberofLanes'] = df['NumberofLanes'].fillna(df['NumberofLanes'].median() if df['NumberofLanes'].notna().any() else 1)

        return df

In [11]:
# ==============================================================================
# FEATURE ENGINEERING FUNCTIONS
# ==============================================================================

def create_time_features(df):
    """Create time-based features"""
    df = df.copy()

    df['hour'] = df['timestamp'].dt.hour
    df['dayofweek'] = df['timestamp'].dt.dayofweek
    df['month'] = df['timestamp'].dt.month
    df['quarter'] = df['timestamp'].dt.quarter
    df['dayofyear'] = df['timestamp'].dt.dayofyear
    df['weekofyear'] = df['timestamp'].dt.isocalendar().week.astype(int)

    df['is_weekend'] = df['dayofweek'].isin([5, 6]).astype(int)
    df['is_morning_rush'] = ((df['hour'].between(7, 9)) & (df['is_weekend'] == 0)).astype(int)
    df['is_evening_rush'] = ((df['hour'].between(17, 19)) & (df['is_weekend'] == 0)).astype(int)
    df['is_night'] = df['hour'].isin(list(range(22, 24)) + list(range(0, 6))).astype(int)

    # Cyclical encoding
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['dow_sin'] = np.sin(2 * np.pi * df['dayofweek'] / 7)
    df['dow_cos'] = np.cos(2 * np.pi * df['dayofweek'] / 7)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

    return df

def create_geospatial_features(df):
    """Create geospatial features"""
    df = df.copy()

    def decode_geohash(g):
        try:
            lat, lon = geohash2.decode(g)
            return float(lat), float(lon)
        except:
            return np.nan, np.nan

    unique_gh = df['geohash'].unique()
    gh_to_coord = {g: decode_geohash(g) for g in unique_gh}
    coords = df['geohash'].map(gh_to_coord)

    df['latitude'] = coords.apply(lambda x: x[0] if isinstance(x, tuple) else np.nan)
    df['longitude'] = coords.apply(lambda x: x[1] if isinstance(x, tuple) else np.nan)

    # Geohash at different precisions
    df['geohash_4'] = df['geohash'].str[:4]
    df['geohash_5'] = df['geohash'].str[:5]
    df['geohash_6'] = df['geohash'].str[:6]

    # Distance from center
    center_lat = df['latitude'].median() if df['latitude'].notna().any() else 0
    center_lon = df['longitude'].median() if df['longitude'].notna().any() else 0
    df['dist_from_center'] = np.sqrt(
        (df['latitude'].fillna(center_lat) - center_lat)**2 +
        (df['longitude'].fillna(center_lon) - center_lon)**2
    )

    return df

def create_interaction_features(df):
    """Create interaction features"""
    df = df.copy()

    if 'Temperature' in df.columns:
        df['temp_x_hour'] = df['Temperature'] * df['hour']
        df['temp_x_weekend'] = df['Temperature'] * df['is_weekend']
        df['temp_squared'] = df['Temperature'] ** 2

    if 'Weather' in df.columns:
        weather_map = {'Clear': 0, 'Clouds': 1, 'Rain': 2, 'Snow': 3, 'Storm': 4}
        df['weather_code'] = df['Weather'].map(weather_map).fillna(0).astype(int)
        bad_weather = df['Weather'].isin(['Rain', 'Snow', 'Storm']).astype(int)
        df['bad_weather_x_rush'] = bad_weather * (df['is_morning_rush'] | df['is_evening_rush']).astype(int)

    if 'RoadType' in df.columns:
        road_map = {'Residential': 0, 'Arterial': 1, 'Highway': 2}
        df['road_code'] = df['RoadType'].map(road_map).fillna(0).astype(int)

    if 'NumberofLanes' in df.columns:
        df['NumberofLanes'] = pd.to_numeric(df['NumberofLanes'], errors='coerce').fillna(1)
        df['lanes_x_temperature'] = df['NumberofLanes'] * df['Temperature']
        df['lanes_x_hour'] = df['NumberofLanes'] * df['hour']

    return df

def encode_categorical_features(df, fit=False, encoder_dict=None):
    """Encode categorical features"""
    df = df.copy()
    categorical_cols = ['geohash_4', 'geohash_5', 'geohash_6']

    if fit:
        encoder_dict = {}
        for col in categorical_cols:
            if col in df.columns:
                le = LabelEncoder()
                df[col + '_encoded'] = le.fit_transform(df[col].astype(str))
                encoder_dict[col] = le
        return df, encoder_dict
    else:
        for col in categorical_cols:
            if col in df.columns and encoder_dict and col in encoder_dict:
                le = encoder_dict[col]
                df[col + '_encoded'] = df[col].astype(str).apply(
                    lambda x: le.transform([x])[0] if x in le.classes_ else -1
                )
        return df, encoder_dict



In [12]:
# ==============================================================================
# ADVANCED ENSEMBLE CLASS (FIXED)
# ==============================================================================

class AdvancedEnsemble:
    """Advanced ensemble with multiple models and stacking"""

    def __init__(self, seed=SEED):
        self.seed = seed
        self.models = {}
        self.meta_model = None
        self.feature_cols = None

    def get_base_models(self):
        """Define base models for ensemble"""
        models = {
            'lightgbm': lgb.LGBMRegressor(
                n_estimators=500, max_depth=8, num_leaves=31,
                learning_rate=0.05, random_state=self.seed, n_jobs=-1, verbose=-1
            ),
            'catboost': CatBoostRegressor(
                iterations=500, learning_rate=0.05, depth=6,
                random_seed=self.seed, verbose=0, thread_count=4
            ),
            'random_forest': RandomForestRegressor(
                n_estimators=300, max_depth=12, min_samples_split=10,
                random_state=self.seed, n_jobs=-1
            ),
            'gradient_boost': GradientBoostingRegressor(
                n_estimators=300, max_depth=6, learning_rate=0.05,
                random_state=self.seed
            ),
            'extra_trees': ExtraTreesRegressor(
                n_estimators=200, max_depth=10, random_state=self.seed, n_jobs=-1
            ),
            'adaboost': AdaBoostRegressor(
                n_estimators=200, learning_rate=0.05, random_state=self.seed
            ),
        }

        # Add XGBoost if available
        if XGB_AVAILABLE:
            models['xgboost'] = xgb.XGBRegressor(
                n_estimators=500, max_depth=6, learning_rate=0.05,
                random_state=self.seed, n_jobs=-1
            )

        return models

    def train_base_models(self, X_train, y_train, X_val, y_val):
        """Train all base models and return predictions"""
        base_models = self.get_base_models()
        predictions = {}
        scores = {}

        print("\n" + "="*60)
        print("TRAINING BASE MODELS")
        print("="*60)

        for name, model in base_models.items():
            print(f"\nTraining {name}...")

            try:
                model.fit(X_train, y_train)
                preds = model.predict(X_val)
                r2 = r2_score(y_val, preds)
                scores[name] = r2
                predictions[name] = preds
                self.models[name] = model
                print(f"  {name} - Validation R²: {r2:.4f} | Score: {max(0, 100*r2):.2f}")
            except Exception as e:
                print(f"  {name} failed: {e}")
                scores[name] = 0
                predictions[name] = np.mean(y_train) * np.ones_like(y_val)

        return predictions, scores

    def train_voting_ensemble(self, X_train, y_train, X_val, y_val, predictions):
        """Train voting ensemble (weighted average)"""
        print("\n" + "="*60)
        print("TRAINING VOTING ENSEMBLE")
        print("="*60)

        # Calculate optimal weights based on validation performance
        scores = [r2_score(y_val, predictions[name]) for name in predictions]
        weights = np.array(scores) / (sum(scores) + 1e-8)

        weighted_preds = np.zeros(len(y_val))
        for i, name in enumerate(predictions.keys()):
            weighted_preds += weights[i] * predictions[name]

        weighted_r2 = r2_score(y_val, weighted_preds)
        print(f"\nWeighted Voting Ensemble R²: {weighted_r2:.4f}")
        print(f"Ensemble Score: {max(0, 100*weighted_r2):.2f}")

        # Store weights
        self.voting_weights = {name: weights[i] for i, name in enumerate(predictions.keys())}

        return weighted_r2

    def train_stacking_ensemble(self, X_train, y_train, X_val, y_val, predictions):
        """Train stacking ensemble with meta-learner"""
        print("\n" + "="*60)
        print("TRAINING STACKING ENSEMBLE")
        print("="*60)

        # Prepare meta-features
        meta_features = np.column_stack([predictions[name] for name in predictions])

        # Train multiple meta-learners
        meta_models = {
            'ridge': Ridge(alpha=1.0, random_state=self.seed),
            'lasso': Lasso(alpha=0.01, random_state=self.seed),
            'elastic': ElasticNet(alpha=0.01, l1_ratio=0.5, random_state=self.seed),
            'linear': LinearRegression()
        }

        best_meta = None
        best_r2 = -np.inf
        best_name = None

        for name, model in meta_models.items():
            try:
                model.fit(meta_features, y_val)
                preds = model.predict(meta_features)
                r2 = r2_score(y_val, preds)
                print(f"  {name} Meta-Learner R²: {r2:.4f}")

                if r2 > best_r2:
                    best_r2 = r2
                    best_meta = model
                    best_name = name
            except Exception as e:
                print(f"  {name} failed: {e}")

        self.meta_model = best_meta
        self.meta_features_names = list(predictions.keys())

        print(f"\nBest Meta-Learner: {best_name}")
        print(f"Stacking Ensemble R²: {best_r2:.4f}")
        print(f"Ensemble Score: {max(0, 100*best_r2):.2f}")

        return best_r2

    def train_blending_ensemble(self, X_train, y_train, X_val, y_val, predictions):
        """Train blending ensemble (optimized linear combination)"""
        print("\n" + "="*60)
        print("TRAINING BLENDING ENSEMBLE")
        print("="*60)

        from scipy.optimize import minimize

        meta_features = np.column_stack([predictions[name] for name in predictions])

        def objective(weights):
            weights = np.array(weights)
            weights = weights / (weights.sum() + 1e-8)
            ensemble_pred = np.dot(meta_features, weights)
            return -r2_score(y_val, ensemble_pred)  # Negative for minimization

        n_models = len(predictions)
        initial_weights = np.ones(n_models) / n_models
        bounds = [(0, 1) for _ in range(n_models)]
        constraints = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}

        try:
            result = minimize(objective, initial_weights, bounds=bounds,
                             constraints=constraints, method='SLSQP')
            optimal_weights = result.x / (result.x.sum() + 1e-8)
        except:
            # Fallback to equal weights
            optimal_weights = np.ones(n_models) / n_models

        self.blending_weights = {name: optimal_weights[i] for i, name in enumerate(predictions.keys())}

        ensemble_pred = np.dot(meta_features, optimal_weights)
        blending_r2 = r2_score(y_val, ensemble_pred)

        print(f"\nOptimal weights:")
        for name, weight in self.blending_weights.items():
            if weight > 0.01:
                print(f"  {name}: {weight:.3f}")

        print(f"\nBlending Ensemble R²: {blending_r2:.4f}")
        print(f"Ensemble Score: {max(0, 100*blending_r2):.2f}")

        return blending_r2

    def predict(self, X_test, method='stacking'):
        """Generate predictions using the specified ensemble method"""
        print(f"\nGenerating predictions using {method} ensemble...")

        # Get base model predictions
        base_preds = {}
        for name, model in self.models.items():
            try:
                base_preds[name] = model.predict(X_test)
            except:
                base_preds[name] = np.zeros(len(X_test))

        if method == 'voting':
            # Weighted average
            final_preds = np.zeros(len(X_test))
            total_weight = 0
            for name, pred in base_preds.items():
                weight = self.voting_weights.get(name, 1/len(base_preds))
                final_preds += weight * pred
                total_weight += weight
            final_preds = final_preds / total_weight

        elif method == 'stacking':
            # Stacking with meta-learner
            if self.meta_model is not None and hasattr(self, 'meta_features_names'):
                meta_features = np.column_stack([base_preds[name] for name in self.meta_features_names if name in base_preds])
                if meta_features.shape[1] > 0:
                    final_preds = self.meta_model.predict(meta_features)
                else:
                    final_preds = np.mean(list(base_preds.values()), axis=0)
            else:
                final_preds = np.mean(list(base_preds.values()), axis=0)

        elif method == 'blending':
            # Blending with optimal weights
            final_preds = np.zeros(len(X_test))
            total_weight = 0
            for name, pred in base_preds.items():
                weight = self.blending_weights.get(name, 0)
                final_preds += weight * pred
                total_weight += weight
            if total_weight > 0:
                final_preds = final_preds / total_weight
            else:
                final_preds = np.mean(list(base_preds.values()), axis=0)

        else:
            # Simple average
            final_preds = np.mean(list(base_preds.values()), axis=0)

        return final_preds



In [13]:
# ==============================================================================
# MAIN FUNCTION
# ==============================================================================

def main():
    """Complete pipeline with advanced ensemble"""

    print("\n" + "="*60)
    print("STEP 1: LOADING DATA")
    print("="*60)

    processor = DataProcessor()
    df_train_raw, df_test_raw = processor.load_data(
        "dataset_extracted/dataset/train.csv",
        "dataset_extracted/dataset/test.csv"
    )

    print(f"Train shape: {df_train_raw.shape}")
    print(f"Test shape: {df_test_raw.shape}")

    # ======================================================================
    # STEP 2: PREPROCESSING
    # ======================================================================
    print("\n" + "="*60)
    print("STEP 2: PREPROCESSING")
    print("="*60)

    df_train_raw = processor.fix_data_types(df_train_raw)
    df_test_raw = processor.fix_data_types(df_test_raw)

    df_train_raw = processor.parse_timestamps(df_train_raw)
    df_test_raw = processor.parse_timestamps(df_test_raw)

    df_train_raw = processor.handle_missing_values(df_train_raw)
    df_test_raw = processor.handle_missing_values(df_test_raw, df_train_raw)

    print(f"Train after preprocessing: {df_train_raw.shape}")
    print(f"Test after preprocessing: {df_test_raw.shape}")

    # ======================================================================
    # STEP 3: CREATE FEATURES
    # ======================================================================
    print("\n" + "="*60)
    print("STEP 3: CREATING FEATURES")
    print("="*60)

    # Create features
    df_train = create_time_features(df_train_raw)
    df_train = create_geospatial_features(df_train)
    df_train = create_interaction_features(df_train)

    df_test = create_time_features(df_test_raw)
    df_test = create_geospatial_features(df_test)
    df_test = create_interaction_features(df_test)

    # Encode categorical features
    df_train, encoder_dict = encode_categorical_features(df_train, fit=True)
    df_test, _ = encode_categorical_features(df_test, fit=False, encoder_dict=encoder_dict)

    # Drop original categorical columns
    categorical_columns_to_drop = ['RoadType', 'Weather', 'Landmarks', 'LargeVehicles',
                                    'geohash', 'geohash_4', 'geohash_5', 'geohash_6']
    for col in categorical_columns_to_drop:
        if col in df_train.columns:
            df_train = df_train.drop(columns=[col])
        if col in df_test.columns:
            df_test = df_test.drop(columns=[col])

    # Get feature columns
    feature_cols = [c for c in df_train.columns
                    if c not in ['demand', 'Index', 'timestamp', 'latitude', 'longitude']
                    and df_train[c].dtype != 'object'
                    and 'lag' not in c
                    and 'roll' not in c]

    print(f"Total features: {len(feature_cols)}")

    # Prepare data
    X = df_train[feature_cols].values.astype(np.float32)
    y = df_train['demand'].values.astype(np.float32)
    X_test = df_test[feature_cols].values.astype(np.float32)

    # Clean NaNs
    X = np.nan_to_num(X, nan=0, posinf=0, neginf=0)
    X_test = np.nan_to_num(X_test, nan=0, posinf=0, neginf=0)

    print(f"\nFinal training shape: {X.shape}")
    print(f"Final test shape: {X_test.shape}")

    # ======================================================================
    # STEP 4: TRAIN/VALIDATION SPLIT
    # ======================================================================
    print("\n" + "="*60)
    print("STEP 4: TRAIN/VALIDATION SPLIT")
    print("="*60)

    # Use last 20% for validation (time series order)
    split_idx = int(len(X) * 0.8)
    X_train = X[:split_idx]
    y_train = y[:split_idx]
    X_val = X[split_idx:]
    y_val = y[split_idx:]

    print(f"Training samples: {len(X_train)}")
    print(f"Validation samples: {len(X_val)}")

    # ======================================================================
    # STEP 5: TRAIN ADVANCED ENSEMBLE
    # ======================================================================
    print("\n" + "="*60)
    print("STEP 5: TRAINING ADVANCED ENSEMBLE")
    print("="*60)

    ensemble = AdvancedEnsemble(seed=SEED)

    # Train base models
    base_predictions, base_scores = ensemble.train_base_models(X_train, y_train, X_val, y_val)

    # Train different ensemble methods
    voting_r2 = ensemble.train_voting_ensemble(X_train, y_train, X_val, y_val, base_predictions)
    stacking_r2 = ensemble.train_stacking_ensemble(X_train, y_train, X_val, y_val, base_predictions)
    blending_r2 = ensemble.train_blending_ensemble(X_train, y_train, X_val, y_val, base_predictions)

    # ======================================================================
    # STEP 6: SELECT BEST ENSEMBLE METHOD
    # ======================================================================
    print("\n" + "="*60)
    print("STEP 6: SELECTING BEST ENSEMBLE")
    print("="*60)

    # Ensure all values are floats
    ensemble_results = {
        'Voting Ensemble': float(voting_r2) if voting_r2 is not None else 0,
        'Stacking Ensemble': float(stacking_r2) if stacking_r2 is not None else 0,
        'Blending Ensemble': float(blending_r2) if blending_r2 is not None else 0
    }

    best_method = max(ensemble_results, key=ensemble_results.get)
    best_r2 = ensemble_results[best_method]

    print(f"\n{'='*50}")
    print("ENSEMBLE COMPARISON")
    print(f"{'='*50}")
    for method, r2 in ensemble_results.items():
        print(f"{method}: R² = {r2:.4f} | Score = {max(0, 100*r2):.2f}")
    print(f"\n🏆 Best Method: {best_method}")
    print(f"Best Validation R²: {best_r2:.4f}")
    print(f"Best Expected Score: {max(0, 100*best_r2):.2f}")

    # ======================================================================
    # STEP 7: GENERATE TEST PREDICTIONS
    # ======================================================================
    print("\n" + "="*60)
    print("STEP 7: GENERATING TEST PREDICTIONS")
    print("="*60)

    # Use best ensemble method
    method_map = {'Voting Ensemble': 'voting', 'Stacking Ensemble': 'stacking', 'Blending Ensemble': 'blending'}
    final_preds = ensemble.predict(X_test, method=method_map[best_method])

    # Clip to valid range
    final_preds = np.clip(final_preds, 0, y.max())

    print(f"Predictions - Min: {final_preds.min():.4f}, Max: {final_preds.max():.4f}, Mean: {final_preds.mean():.4f}")

    # ======================================================================
    # STEP 8: CREATE SUBMISSION
    # ======================================================================
    print("\n" + "="*60)
    print("STEP 8: CREATING SUBMISSION")
    print("="*60)

    submission = pd.DataFrame({
        'Index': df_test_raw['Index'].values,
        'demand': final_preds
    })

    submission.to_csv("submission_ensemble.csv", index=False)
    print(f"Submission saved: submission_ensemble.csv")
    print(f"Shape: {submission.shape}")
    print(f"Demand stats:\n{submission['demand'].describe()}")

    # ======================================================================
    # FINAL EVALUATION
    # ======================================================================
    print("\n" + "="*60)
    print("FINAL EVALUATION")
    print("="*60)

    expected_score = max(0, 100 * best_r2)

    print(f"\n{'='*50}")
    print(f"🏆 FINAL EXPECTED HACKATHON SCORE: {expected_score:.2f} / 100")
    print(f"{'='*50}")

    if expected_score >= 70:
        print("\n🎯 Grade: EXCELLENT - Strong chance to win!")
    elif expected_score >= 55:
        print("\n👍 Grade: GOOD - Competitive position!")
    elif expected_score >= 40:
        print("\n📊 Grade: AVERAGE - Need improvement")
    else:
        print("\n⚠️ Grade: NEEDS IMPROVEMENT - Check for issues")

    # Save ensemble summary
    summary = pd.DataFrame({
        'Model': list(base_scores.keys()) + list(ensemble_results.keys()),
        'Validation_R2': list(base_scores.values()) + list(ensemble_results.values()),
        'Expected_Score': [max(0, 100*s) for s in list(base_scores.values()) + list(ensemble_results.values())]
    })
    summary.to_csv('ensemble_performance.csv', index=False)
    print("\n✅ Performance summary saved to 'ensemble_performance.csv'")

    print("\n" + "="*60)
    print("PIPELINE COMPLETE!")
    print("="*60)
    print("\n📁 Output files:")
    print("   - submission_ensemble.csv (final predictions)")
    print("   - ensemble_performance.csv (model comparison)")
    print("\n🚀 UPLOAD submission_ensemble.csv TO THE HACKATHON PLATFORM")

    return submission


# ==============================================================================
# RUN MAIN FUNCTION
# ==============================================================================
if __name__ == "__main__":
    submission = main()


STEP 1: LOADING DATA
Train shape: (77299, 11)
Test shape: (41778, 10)

STEP 2: PREPROCESSING
Train after preprocessing: (77299, 11)
Test after preprocessing: (41778, 10)

STEP 3: CREATING FEATURES
Total features: 31

Final training shape: (77299, 31)
Final test shape: (41778, 31)

STEP 4: TRAIN/VALIDATION SPLIT
Training samples: 61839
Validation samples: 15460

STEP 5: TRAINING ADVANCED ENSEMBLE

TRAINING BASE MODELS

Training lightgbm...
  lightgbm - Validation R²: 0.5967 | Score: 59.67

Training catboost...
  catboost - Validation R²: 0.6353 | Score: 63.53

Training random_forest...
  random_forest - Validation R²: 0.6891 | Score: 68.91

Training gradient_boost...
  gradient_boost - Validation R²: 0.6567 | Score: 65.67

Training extra_trees...
  extra_trees - Validation R²: 0.6423 | Score: 64.23

Training adaboost...
  adaboost - Validation R²: 0.6000 | Score: 60.00

Training xgboost...
  xgboost - Validation R²: 0.5954 | Score: 59.54

TRAINING VOTING ENSEMBLE

Weighted Voting Ensem